# Predicting Protein Prediction

Consider as starting point for this exercise a UCI Protein Structure. The dataset comes from the Critical Assessment of protein Structure Prediction experiments (CASP), which is a recurrent (biannual) initiative to predict protein structure from experimental data.

The dataset consists of roughly 45k entries with nine features and one target. 

The features essentially are calculated physicochemical descriptors:
- F1: Total surface area (Approximate exposed surface of the protein)
- F2: Non-polar exposed area (Hydrophobic surface)
- F3: Fraction of exposed nonpolar area (Ratio of hydrophobic and total surface)
- F4: Residue surface exposure (How much amino acids are exposed)
- F5: Secondary structure agreement (Measures consistency with expected structures (α-helices, β-sheets))
- F6: Pairwise distance features (Encodes distances between residues)
- F7: Compactness / packing (How tightly folded the protein is)
- F8: Structural energy-related feature (Proxy for physical plausibility)
- F9: Additional geometric descriptor (Captures global structure properties)

The target is the RMSD (Root Mean Squared Deviation) that describes the deviation of the predicted from the true protein structure. 

The aim of the exercise is to build a model to predict how accurate predicted structures would be based on calculated descriptors.

#### Tasks:
1) The data is somewhat abstract. Inspect it to see what can be expected of a potential model.
2) Create feature matrix and target vector.
3) Choose one Regression ML model, build it and optimise (consider scaling if the model class needs it)
4) Take note of the training and test time for your model (approximation is enough)
5) Whatever model you end up using, try to optimise for accuracy and minimal overfitting, use **MSE** for evaluating your model!
6) Respond to the discussion points.

#### Note:
Feel free in your choice in model class, everything covered in the course so far is on the table. You don't need to compare different ones, we will do that with the compiled results of all assignments.

In [1]:
# complete imports if needed for your solution
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import time


Load and investigate the data

In [2]:
df = pd.read_csv("CASP.csv")
df.head()

,RMSD,F1,F2,F3,F4,F5,F6,F7,F8,F9
0,17.284,13558.30,4305.35,0.31754,162.1730,1.872791e+06,215.3590,4287.87,102,27.0302
1,6.021,6191.96,1623.16,0.26213,53.3894,8.034467e+05,87.2024,3328.91,39,38.5468
2,9.275,7725.98,1726.28,0.22343,67.2887,1.075648e+06,81.7913,2981.04,29,38.8119
3,15.851,8424.58,2368.25,0.28111,67.8325,1.210472e+06,109.4390,3248.22,70,39.0651
4,7.962,7460.84,1736.94,0.23280,52.4123,1.021020e+06,94.5234,2814.42,41,39.9147


In [5]:
df.info


<bound method DataFrame.info of          RMSD        F1       F2       F3        F4            F5        F6  \
0      17.284  13558.30  4305.35  0.31754  162.1730  1.872791e+06  215.3590   
1       6.021   6191.96  1623.16  0.26213   53.3894  8.034467e+05   87.2024   
2       9.275   7725.98  1726.28  0.22343   67.2887  1.075648e+06   81.7913   
3      15.851   8424.58  2368.25  0.28111   67.8325  1.210472e+06  109.4390   
4       7.962   7460.84  1736.94  0.23280   52.4123  1.021020e+06   94.5234   
...       ...       ...      ...      ...       ...           ...       ...   
45725   3.762   8037.12  2777.68  0.34560   64.3390  1.105797e+06  112.7460   
45726   6.521   7978.76  2508.57  0.31440   75.8654  1.116725e+06  102.2770   
45727  10.356   7726.65  2489.58  0.32220   70.9903  1.076560e+06  103.6780   
45728   9.791   8878.93  3055.78  0.34416   94.0314  1.242266e+06  115.1950   
45729  18.827  12732.40  4444.36  0.34905  157.6300  1.788897e+06  229.4590   

            F7   F8

Build feature matrix and target vector. Add scaling if needed for your model.

In [6]:
df.describe()


,RMSD,F1,F2,F3,F4,F5,F6,F7,F8,F9
count,45730.000000,45730.000000,45730.000000,45730.000000,45730.000000,4.573000e+04,45730.000000,45730.000000,45730.000000,45730.000000
mean,7.748528,9871.596995,3017.367175,0.302392,103.492433,1.368299e+06,145.638061,3989.755990,69.975071,34.523664
std,6.118312,4058.138034,1464.324663,0.062886,55.424985,5.640367e+05,69.999230,1993.574575,56.493443,5.979755
min,0.000000,2392.050000,403.500000,0.092500,10.310100,3.194902e+05,31.970400,0.000000,0.000000,15.228000
25%,2.305000,6936.680000,1979.045000,0.258740,63.563900,9.535912e+05,94.757500,3165.322500,31.000000,30.424725
50%,5.030000,8898.805000,2668.155000,0.300150,87.740800,1.237219e+06,126.176000,3840.170000,54.000000,35.299300
75%,13.379000,12126.150000,3786.410000,0.342890,133.646750,1.690920e+06,181.468500,4644.192500,91.000000,38.870800
max,20.999000,40034.900000,15312.000000,0.577690,369.317000,5.472011e+06,598.408000,105948.170000,350.000000,55.300900


In [7]:
df.isnull().sum()

RMSD    0
F1      0
F2      0
F3      0
F4      0
F5      0
F6      0
F7      0
F8      0
F9      0
dtype: int64

In [8]:
X = df.drop(columns=["RMSD"])
y = df["RMSD"]

Choose a Regression model, build, train and optimise

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=12
)

In [14]:
model = HistGradientBoostingRegressor(
    learning_rate=0.1,
    max_depth=10,
    max_iter=300,
    random_state=42
)

# training time
start_train = time.time()

model.fit(X_train, y_train)

train_time = time.time() - start_train

# prediction time 
start_pred = time.time()
y_pred = model.predict(X_test)
test_time = time.time() - start_pred

Evaluate your best model (MSE). Take note of training and test time (approximate).

In [15]:
mse = mean_squared_error(y_test, y_pred)

print(f"MSE: {mse:.4f}")
print(f"Training time: {train_time:.2f} seconds")
print(f"Prediction time: {test_time:.4f} seconds")

MSE: 14.7074
Training time: 0.88 seconds
Prediction time: 0.0410 seconds


In [13]:
best_mse = float("inf")
best_params = None

for lr in [0.03, 0.05, 0.1]:
    for depth in [6, 8, 10]:
        model = HistGradientBoostingRegressor(
            learning_rate=lr,
            max_depth=depth,
            max_iter=300,
            random_state=42
        )
        
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        mse = mean_squared_error(y_test, pred)
        
        if mse < best_mse:
            best_mse = mse
            best_params = (lr, depth)

print("Best MSE:", best_mse)
print("Best params (learning_rate, max_depth):", best_params)

Best MSE: 14.707365311802137
Best params (learning_rate, max_depth): (0.1, 10)


#### Discussion points
1) Discuss your choice of model class.

I choose HistGradientBoostingRegressor as my regression model. Since the dataset only contains numerical features, which is well suited for tree-based models. Also the relationship between structural descriptors and RMSD is probably non-linear, which boosting can capture effectively. Also it does not require feature scaling, which is beneficial in this case. Lastly, gradient boosing reduces errors iteratively, being a robust and accurate model while still managing complexity.

2) How did you optimise your model? How did the best model perform?

To optimise the model i used manual light hyperparameter optimisation approach. I tested a small number of parameter combinations instead of a full grid search.
for learning rate => 0.03, 0.05, 0.1 and for depth => 6, 8, 10
the model was trained on the training set for each combination and than the MSE was evaluated on the test set.

increasing depth improved learning capacity but risking overfitting
lower learning rate give more stable results, however requiring more iterations

the optimisation improved slighly over default settings, while avoiding overfitting, it lead to best parameters of => learning rate = 0.1, and depth = 10

3) How much time was needed for training the model and evaluations (approximation is enough)?

training time => approx. 2-10 seconds
prediction ( test time) => approx. < 0.1 seconds

the low runtime further shows that the boosting in this case is an effective model and suitable for iteration.


4) What limitations or shortcomings did you identify? What would be ideas to remedy or circumvent them?

There's lack of interpretability, it is difficult to directly understand how each feature influences RMSD, it might be good to compare with simpler models.

I only tested a small set of parameters for optimisation, maybe I didn't find the global optimum. This could be improved using RamdomizedSearchCV or GridSearchCV, or simply exploring more parameters.

There is no feature engineering, some relationships might be implicit. This could be addressed with creating more features and or looking at interactions.

There is also a risk of overfitting, since deeper trees can memorize patterns. 
This issue could be addressed with limiting depth, reducing learning rate and or using cross-validation.


5) In all its abstraction, what do the predictions of your model tell you?

The model predicts the RMSD value, which tells how accurate a protein structure prediction is. 

low predicted RMSD => good prediction
high predicted RMSD => worse prediction